# Built-in Data Structures from Scratch

## Lists, Arrays & Slicing Internals

### Core Mechanics & Theory

To write high-performance Python and prepare for low-level tensor/C-array manipulation, you need to understand how Python lists work in memory.

### 1. Lists are Dynamic Arrays of Pointers

A Python list is not a linked list and not a flat array of raw data. It is a C array of pointers to PyObjects:

```
PyObject** ob_item
list variable ---> [ PyListObject Header | ob_size | allocated ]
                                           │
                                           ▼ (Array of pointers)
                                  [ ptr0 | ptr1 | ptr2 | ptr3 ]
                                     │      │      │      │
                                     ▼      ▼      ▼      ▼
                                   PyObj  PyObj  PyObj  PyObj
```

### 2. Over-Allocation & Amortized $O(1)$ Append

When a list grows, CPython does not allocate space for just one element (which would make `.append()` $O(N)$ due to constant realloc calls). It allocates extra headroom using an over-allocation growth formula:

$$\text{new\_allocated} \approx \text{new\_size} + (\text{new\_size} \gg 3) + (\text{new\_size} < 9 \,?\, 3 : 6)$$

**Time Complexity:**
- `.append()`: $O(1)$ amortized
- `.insert(0, val)` or `.pop(0)`: $O(N)$ because every pointer in the internal array must be shifted in memory

### 3. Slice Notation Internals: [start:stop:step]

A slice `s[start:stop:step]` creates a slice object: `slice(start, stop, step)`.

**Step arithmetic:**
- **Positive step**: starts at start, increments by step, stops before reaching stop
- **Negative step**: starts at start, decrements by abs(step), stops before reaching stop (e.g., `[::-1]` flips the pointer array)

**Important:** Slicing a list always creates a new list containing a shallow copy of the original pointer addresses.

### 2. Over-Allocation & Amortized $O(1)$ Append

When a list grows, CPython does not allocate space for just one element (which would make `.append()` $O(N)$ due to constant realloc calls). It allocates extra headroom using an over-allocation growth formula:

$$\text{new\_allocated} \approx \text{new\_size} + (\text{new\_size} \gg 3) + (\text{new\_size} < 9 \,?\, 3 : 6)$$

**Time Complexity:**
- `.append()`: $O(1)$ amortized
- `.insert(0, val)` or `.pop(0)`: $O(N)$ because every pointer in the internal array must be shifted in memory

### 3. Slice Notation Internals: [start:stop:step]

A slice `s[start:stop:step]` creates a slice object: `slice(start, stop, step)`.

**Step arithmetic:**
- **Positive step**: starts at start, increments by step, stops before reaching stop
- **Negative step**: starts at start, decrements by abs(step), stops before reaching stop (e.g., `[::-1]` flips the pointer array)

**Important:** Slicing a list always creates a new list containing a shallow copy of the original pointer addresses.

## Constraints

Implement these exercises without using built-in helper methods like `.reverse()`, `.rotate()`, or importing `collections.deque`.

## Exercise 1: In-Place Array Reversal (Two-Pointer Technique)

Write a function `reverse_inplace(arr: list)` that reverses a list in-place ($O(1)$ auxiliary space) using manual index pointers `left` and `right`.

**Requirements:**
- Do not allocate a new list or use `arr[::-1]`
- Mutate the original list directly using pointer swapping: `arr[left], arr[right] = arr[right], arr[left]`

In [1]:
def reverse_inplace(arr:list, start=0, end=None):
    left = start
    right = len(arr)-1 if end==None else end
    print("before", arr)
    while left<right:
        arr[left], arr[right] = arr[right], arr[left]
        left+=1
        right-=1
    print("after",arr)
    return arr

In [2]:
reverse_inplace([3,2,1])

before [3, 2, 1]
after [1, 2, 3]


[1, 2, 3]

In [3]:
reverse_inplace([1,2,3,4,5,6],2,5)

before [1, 2, 3, 4, 5, 6]
after [1, 2, 6, 5, 4, 3]


[1, 2, 6, 5, 4, 3]

## Exercise 2: In-Place Array Rotation (The 3-Step Reversal Algorithm)

Rotate a list to the right by `k` steps in-place with $O(1)$ extra space.

### Why the 3-Reversal Approach?

**Time Complexity Comparison:**
- Repeatedly shifting elements: $O(n \times k)$ time
- The 3-reversal method: $O(n)$ time and $O(1)$ extra space

**Example with k = 3:**

| Approach | Work |
|----------|------|
| Shifting | Move elements 3 times → more work |
| Reversal | Each element is swapped only a constant number of times → $O(n)$ |

So the reversal algorithm is mainly about better time complexity while staying in-place.

In [4]:
def shift_array(arr:list,k:int):
    
    k = k % len(arr)
    
    arr = reverse_inplace(arr)
    
    arr = reverse_inplace(arr,0,k-1)
    
    arr = reverse_inplace(arr,k)
    
    print(arr)

In [5]:
shift_array([1,2,3,4,5,6,7],3)

before [1, 2, 3, 4, 5, 6, 7]
after [7, 6, 5, 4, 3, 2, 1]
before [7, 6, 5, 4, 3, 2, 1]
after [5, 6, 7, 4, 3, 2, 1]
before [5, 6, 7, 4, 3, 2, 1]
after [5, 6, 7, 1, 2, 3, 4]
[5, 6, 7, 1, 2, 3, 4]


## Exercise 3: Manual N-Dimensional Chunking via Slicing

Write a function `chunk_tensor_1d(data: list, chunk_size: int, step: int)` that splits a 1D list into overlapping or non-overlapping windows using manual slice arithmetic.

### Example:

```python
data = list(range(10))
chunk_tensor_1d(data, chunk_size=4, step=2)
```

**Expected output:**
```
[[0, 1, 2, 3], [2, 3, 4, 5], [4, 5, 6, 7], [6, 7, 8, 9]]
```

### Requirements:

Ensure the function cleanly handles truncating any trailing window that does not reach a full `chunk_size`.

In [ ]:
def chunk_overlap_arr(arr : list, chunk:int, step : int):
    new_arr=[]
    i=0
    while i<=len(arr)-chunk:
        new_arr.append(arr[i:i+chunk])
        i+=step
    print(new_arr)
        

In [ ]:
chunk_overlap_arr([1,2,3,4,5,6,7,8,10],3,1)

[[1, 2, 3], [2, 3, 4], [3, 4, 5], [4, 5, 6], [5, 6, 7], [6, 7, 8], [7, 8, 10]]


## Dictionaries & Hash Tables Under the Hood

### Core Mechanics & Theory

CPython's `dict` is a compact, cache-friendly hash table with insertion ordering.

### 1. The Compact Hash Table Layout

Before Python 3.6, dicts allocated large sparse tables containing hash, key pointer, and value pointer, wasting memory on empty rows.

Modern CPython splits the dict into two arrays:

- **Indices Array (Sparse Array)**: A compact hash-indexed array storing offsets into the entries table (e.g., `[-1, 0, -1, 1, -1]`)
- **Entries Array (Dense Array)**: A packed array appending elements in insertion order: `[[hash0, key_ptr0, val_ptr0], [hash1, key_ptr1, val_ptr1], ...]`

```
Hash("key") % Array_Size ---> [ Indices Array ] ---> Index ---> [ Dense Entries Array ]
                              [ -1,  0,  1 ]                    [ hash, key, val ]
```

### 2. Hashability & Collisions

For an object to serve as a dict key, it must be hashable:

- Must implement `__hash__()` returning an integer that never changes across its lifetime
- Must implement `__eq__()` for equality checks when two different keys generate the same hash slot (collision)
- If two objects are equal (`a == b`), their hashes must be identical (`hash(a) == hash(b)`)

### 3. Resolving Collisions: Open Addressing

CPython uses open addressing with a pseudo-random perturbation sequence ($O(1)$ average, $O(N)$ worst-case):

$$\text{next\_index} = ((5 \times \text{index}) + 1 + \text{perturb}) \pmod{\text{size}}$$

## Hands-On Coding Exercises

Implement these without using `collections.defaultdict` or `collections.Counter`.

In [10]:
class SimpleDict:
    def __init__(self, size=8):
        self.size = size
        self.buckets = [None] * size
    
    def set(self, key, value):
        slot = hash(key) % self.size
        
        self.buckets[slot] = [key, value]
    
    def get(self, key):
        slot = hash(key) % self.size
        
        if self.buckets[slot] is not None and self.buckets[slot][0]==key:
            return self.buckets[slot][1]
        raise KeyError(f"key is not found")
    
d= SimpleDict()
d.set("name","Alex")
d.set("gpu","RTX")

print(d.get("name"))
print(d.get("gpu"))

Alex
RTX


## Exercise 1: Hash Map Collision & Equality Probe

Create a class `HashProbeKey` that takes an integer `id` and a string `label`.

### Requirements:

- Override `__hash__()` to return `self.id % 4` (this forces intentional hash collisions across keys like 1, 5, 9)
- Override `__eq__()` to check equality on both `self.id` and `self.label`
- Create three keys with the same hash:
  ```python
  k1 = HashProbeKey(1, "alpha")
  k2 = HashProbeKey(5, "beta")
  k3 = HashProbeKey(9, "gamma")
  ```
- Store them in a standard Python dictionary: `d = {k1: "Value 1", k2: "Value 2", k3: "Value 3"}`
- Verify that retrieving `d[k1]`, `d[k2]`, and `d[k3]` returns the correct individual values even though their hashes are identical

In [23]:
class HashProbeKey:
    def __init__(self, id, label):
        self.id = id
        self.label = label
        
    def __hash__(self):
        
        return self.id % 4
    
    def __eq__(self, other_class_obj):
        if self.id == other_class_obj.id and self.label == other_class_obj.label:
            return True
        return False
    
k1 = HashProbeKey(1, "alpha")
k2 = HashProbeKey(5, "beta")
k3 = HashProbeKey(9, "gamma")

print(k1.__hash__(), k2.__hash__(), k3.__hash__())
#----> same hash but

print(k1.label, k2.label, k3.label)

print(k1.__eq__(k2))



#now lets test an actual dict
d = {k1:"alpha",k2:"beta",k3:"gamma"}
print(d[k3])
print(d[k2])
        

1 1 1
alpha beta gamma
False
gamma
beta


## Exercise 2 (Extended): Manual Collision Handling with Linear Probing

In [24]:
class HashProbeKey:
    def __init__(self, id, label):
        self.id = id
        self.label = label

    def __hash__(self):
        # Intentionally creates collisions
        return self.id % 4

    def __eq__(self, other):
        if not isinstance(other, HashProbeKey):
            return False

        return self.id == other.id and self.label == other.label


class SimpleHashMap:
    def __init__(self, size=8):
        self.size = size

        # Each slot will contain:
        # [hash, key, value]
        self.buckets = [None] * size

    def probe(self, current_slot):
        # Linear probing:
        # 0 -> 1 -> 2 -> 3 -> ... -> 7 -> 0
        return (current_slot + 1) % self.size

    def set(self, key, value):
        key_hash = hash(key)

        # Find the initial position
        slot = key_hash % self.size

        while True:

            # Empty slot -> insert
            if self.buckets[slot] is None:
                self.buckets[slot] = [key_hash, key, value]
                return

            stored_hash, stored_key, stored_value = self.buckets[slot]

            # Same key already exists -> update value
            if stored_hash == key_hash and stored_key == key:
                self.buckets[slot] = [key_hash, key, value]
                return

            # Collision -> probe next slot
            slot = self.probe(slot)

    def get(self, key):
        key_hash = hash(key)

        # Start at the same position we used during insertion
        slot = key_hash % self.size

        while True:

            # Empty slot means key was never inserted
            if self.buckets[slot] is None:
                raise KeyError(key)

            stored_hash, stored_key, stored_value = self.buckets[slot]

            # Found it
            if stored_hash == key_hash and stored_key == key:
                return stored_value

            # Collision -> keep probing
            slot = self.probe(slot)


# -----------------------------------
# Test
# -----------------------------------

k1 = HashProbeKey(1, "alpha")
k2 = HashProbeKey(5, "beta")
k3 = HashProbeKey(9, "gamma")

print("Hashes:")
print(hash(k1))
print(hash(k2))
print(hash(k3))

d = SimpleHashMap()

d.set(k1, "Value 1")
d.set(k2, "Value 2")
d.set(k3, "Value 3")

print("\nValues:")
print(d.get(k1))
print(d.get(k2))
print(d.get(k3))

print("\nBuckets:")
for i, bucket in enumerate(d.buckets):
    print(i, bucket)

Hashes:
1
1
1

Values:
Value 1
Value 2
Value 3

Buckets:
0 None
1 [1, <__main__.HashProbeKey object at 0x000001C389D33B50>, 'Value 1']
2 [1, <__main__.HashProbeKey object at 0x000001C389D33A60>, 'Value 2']
3 [1, <__main__.HashProbeKey object at 0x000001C389D33730>, 'Value 3']
4 None
5 None
6 None
7 None


## Exercise 3: Inverted Index & Frequency Map

Given a list of document strings:

```python
docs = [
    "cuda kernel matrix multiplication",
    "matrix memory layout contiguous",
    "cuda memory stride allocation"
]
```

Write a function `build_inverted_index(docs)` that loops through the documents and builds a dictionary from scratch mapping each unique word to:

- `total_count`: Total occurrences across all documents
- `doc_ids`: A list of document indices (0, 1, 2) containing that word (without duplicates in the list)

### Expected output shape:

```python
{
    "cuda": {"total_count": 2, "doc_ids": [0, 2]},
    "kernel": {"total_count": 1, "doc_ids": [0]},
    "matrix": {"total_count": 2, "doc_ids": [0, 1]},
    ...
}
```

In [32]:
docs = [
    "cuda kernel matrix multiplication",
    "matrix memory layout contiguous",
    "cuda memory stride allocation"
]

def create_map(docs: list):
    map={}
    for i,doc in enumerate(docs):
        words = doc.strip().split(" ")
        for word in words:
            if word in map:
                
                map[word]["total_count"]+=1
                if i in map[word]["doc_ids"]:
                    continue
                else:
                    map[word]["doc_ids"].append(i)
            else:
                
                map[word]={
                    "total_count":1,
                    "doc_ids":[i]
                }
    print(map)
        

In [33]:
create_map(docs)

{'cuda': {'total_count': 2, 'doc_ids': [0, 2]}, 'kernel': {'total_count': 1, 'doc_ids': [0]}, 'matrix': {'total_count': 2, 'doc_ids': [0, 1]}, 'multiplication': {'total_count': 1, 'doc_ids': [0]}, 'memory': {'total_count': 2, 'doc_ids': [1, 2]}, 'layout': {'total_count': 1, 'doc_ids': [1]}, 'contiguous': {'total_count': 1, 'doc_ids': [1]}, 'stride': {'total_count': 1, 'doc_ids': [2]}, 'allocation': {'total_count': 1, 'doc_ids': [2]}}


## Exercise 4: Two-Sum via Hash Map ($O(N)$ Time)

Given an array of integers and a target sum, find the indices of the two numbers that add up to the target.

### Example:
```
nums = [2, 7, 11, 15], target = 9 → Returns (0, 1)
```

### Requirements:

- **Rule**: Do NOT use nested loops ($O(N^2)$)
- **Approach**:
  1. Create an empty dictionary `seen = {}`
  2. Iterate through `nums` with `enumerate()`
  3. Calculate `complement = target - num`
  4. If `complement` is in `seen`, return `(seen[complement], current_index)`
  5. Otherwise, store `seen[num] = current_index`

In [41]:
nums = [2, 7, 11, 3, 15]
target = 10

def two_sum(nums : list, target: int):
    d={}
    for i, val in enumerate(nums):
        remaining = target - val
        if remaining in d:
            return (d[remaining],i)
        else:
            d[val]=i

print(two_sum(nums, target))

(1, 3)


## Sets, Tuples & Extended Unpacking

### 1. Sets Under the Hood

A set in CPython is essentially a dictionary with keys only and no values (no value pointer in the entry table).

### 2. Set Operations

- **Intersection**: `s1 & s2`
- **Difference**: `s1 - s2` (elements in s1 but not s2)
- **Symmetric Difference**: `s1 ^ s2` (elements in either s1 or s2, but not both)
- **frozenset**: An immutable, hashable version of a set that can be used as a dictionary key or stored inside another set

### 3. Tuples: Immutability & Struct-Like Invariants

A tuple is a fixed-size contiguous array of `PyObject*` pointers.

**Key properties:**
- Because they are immutable, CPython optimizes them aggressively via freelists (reusing allocated memory for small tuples up to 20 elements)
- A tuple is only hashable if every element inside it is hashable (e.g., `(1, 2, [3, 4])` will raise `TypeError: unhashable type: 'list'`)

### 4. Extended Star Unpacking (*rest)

Python allows unpacking sequences into variables dynamically without manual index slicing:

```python
data = [10, 20, 30, 40, 50]

first, *middle, last = data
# first = 10, middle = [20, 30, 40], last = 50

*head, tail = data
# head = [10, 20, 30, 40], tail = 50
```

## Hands-On Coding Exercises

## Exercise 1: GPU Kernel Dependency Resolver (Set Algebra)

You have two GPU device contexts with required compute features:

```python
ctx_a_features = {"fp16", "tensor_cores", "unified_memory", "int8"}
ctx_b_features = {"fp16", "tensor_cores", "flash_attention", "sparse_matmul"}
```

### Requirements:

Write functions using pure set operators (`|`, `&`, `-`, `^`) to return:

- **Union**: All features supported across both devices
- **Intersection**: Common features supported by both devices
- **Difference**: Features exclusive to ctx_a
- **Symmetric Difference**: Features supported by either device, but not both

In [ ]:
ctx_a_features = {"fp16", "tensor_cores", "unified_memory", "int8"}
ctx_b_features = {"fp16", "tensor_cores", "flash_attention", "sparse_matmul"}

In [42]:
ctx_a_features = {"fp16", "tensor_cores", "unified_memory", "int8"}

ctx_b_features = {"fp16", "tensor_cores", "flash_attention", "sparse_matmul"}


def feature_analysis(ctx_a, ctx_b):

    # 1. All features from both devices
    union = ctx_a | ctx_b

    # 2. Features common to both
    intersection = ctx_a & ctx_b

    # 3. Features only in ctx_a
    difference = ctx_a - ctx_b

    # 4. Features in either, but NOT both
    symmetric_difference = ctx_a ^ ctx_b

    return union, intersection, difference, symmetric_difference


union, common, exclusive_a, exclusive = feature_analysis(
    ctx_a_features,
    ctx_b_features
)

print("Union:", union)
print("Intersection:", common)
print("Exclusive to A:", exclusive_a)
print("Symmetric Difference:", exclusive)

Union: {'fp16', 'tensor_cores', 'flash_attention', 'unified_memory', 'sparse_matmul', 'int8'}
Intersection: {'tensor_cores', 'fp16'}
Exclusive to A: {'unified_memory', 'int8'}
Symmetric Difference: {'flash_attention', 'unified_memory', 'sparse_matmul', 'int8'}


## Exercise 3: Stream Frame Header Unpacker (Star Unpacking)

You receive raw serialized stream frames formatted as variable-length lists:

```python
packet1 = ["START", 0xAA, 0x10, 0x20, 0x30, 0xFF, "END"]
packet2 = ["START", 0xBB, 0x55, "END"]
```

### Requirements:

Write a function `unpack_frame(packet: list)` that uses extended star unpacking (`*payload`) in a single line to extract:

- `header` ("START")
- `command_id` (the first hex byte)
- `payload` (all middle data bytes as a list)
- `footer` ("END")

**Validation & Return:**
- Validate that `header == "START"` and `footer == "END"`
- Return a dictionary: `{"cmd": command_id, "data": payload}`

In [45]:
packet1 = ["START", 0xAA, 0x10, 0x20, 0x30, 0xFF, "END"]
packet2 = ["START", 0xBB, 0x55, "END"]

def extract_payload(arr : list):
    start, command, *payload, end = arr
    
    if start !='START' or end != 'END':
        raise ValueError("value not in format")
    
    return {
        "command_id":command,
        "payload":payload
    }

print(extract_payload(packet1))
print(extract_payload(packet2))

{'command_id': 170, 'payload': [16, 32, 48, 255]}
{'command_id': 187, 'payload': [85]}
